# Simple E-NAS

## Problem statment
Finding the optimum number of neurons in hidden layers for the model that recognizes digits.

## Data
In this solution one individual is a list of lists. Each list represent one hidden layer, and each layer(list) contains 0 and 1 that represents bits that will give us number of neurons in each layer.
We will try to create data for 2 layers and maksimum 255 neurons.

Example:
```
individual = [
    [  # first layer
        0, 0, 0, 0, 0, 1, 0, 1  # bit representation of number of neural inputs
    ],
    [  # second layer
        0, 0, 0, 0, 0, 1, 0, 1  # bit representation of number of neural inputs
    ]
]
```

Note:

In number of layers we will not counting output layer due to it always need to have 10 outputs.
Allso we will not include input layer, due to it's depend on image (for dense layers).

In [1]:
seed_data = [
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0]
]

bits_in_number = 8

## GA initialization

In [2]:
from pyeasyga import pyeasyga

ga = pyeasyga.GeneticAlgorithm(seed_data,
                               population_size=15,
                               generations=100,
                               crossover_probability=0.8,
                               mutation_probability=0.05,
                               elitism=True,
                               maximise_fitness=True)

## Define Individual
An individual is an entity of data. COnception for library `pyeasyga` is that the build-in algorithm will create many variations of individuals and in this way, we receive datasets for genetic algorithm.

In [3]:
import random


# define and set function to create a candidate solution representation
def create_individual(data):
    individual = data[:]
    for layer_index in range(len(individual)):
        for bit_index in range(len(individual[layer_index])):
            individual[layer_index][bit_index] = random.randint(0, 1)

    return individual

In [4]:
ga.create_individual = create_individual

## Fitness function

We will use as fitness function:

`f(n1, n2) = MSE(n1, n2) + alfa((n1 + n2)/2(2^N -1))*MSE(n1, n2), 0 < alfa < 1`

Where: 
* n1 - number of neurons in first layer 
* n2 - number of neurons in second layer
* MSE - Mean Squared Error
* 2(2^N-1) - maximum number of neurons in the networ
* N - the number of bits on which the number of neurons in the layer is coded

Reminder:

In number of layers we will not counting output layer due to it always need to have 10 outputs.
Allso we will not include input layer, due to it's depend on image (for dense layers).

In [5]:
from ANN.digit_recognition import (prepare_ann_data, get_model)


train, test = prepare_ann_data()

def count_mse(first_layer_number_of_neurons, second_layer_number_of_neurons):

    model = get_model(
        first_layer_number_of_neurons, 
        second_layer_number_of_neurons, 
        train, 
        test
    )

    mse = 0
    for loss in model.history['val_loss']:
        mse += loss
    
    return mse

Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...: 100%|██████████| 4/4 [00:00<00:00, 73.50 url/s]
Extraction completed...: 0 file [00:00, ? file/s]
Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Dataset mnist downloaded and prepared to ~\tensorflow_datasets\mnist\3.0.1. Subsequent calls will reuse this data.
Cause: Unable to locate the source code of <function normalize_img at 0x000001D79DC9CC10>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function normalize_img at 0x000001D79DC9CC10>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function normalize_img at 0x000001D79DC9CC10>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


In [6]:
from utils import (individual_guard, convert_list_to_int)


def fitness(individual, data):
    fitness = 0

    if individual_guard(individual):  # We need this guard due to data corruption (look to notes)
        first_layer_number_of_neurons = convert_list_to_int(individual[0])
        second_layer_number_of_neurons = convert_list_to_int(individual[1])

        mse = count_mse(first_layer_number_of_neurons, second_layer_number_of_neurons)

        alfa = 0.2
        total_number_of_neurons = first_layer_number_of_neurons + second_layer_number_of_neurons
        maximum_nuerons_in_layer = 2^bits_in_number
        total_maximum_neurons = 2*(maximum_nuerons_in_layer - 1)
        
        punishment_for_large_network_complexity = alfa * (total_number_of_neurons/total_maximum_neurons) * mse
        
        fitness = mse + punishment_for_large_network_complexity

    return fitness


In [7]:
ga.fitness_function = fitness

## Run GA

In [8]:
ga.run()

Epoch 1/6
469/469 [==============================] - 6s 4ms/step - loss: 3.7392 - sparse_categorical_accuracy: 0.0870 - val_loss: 1.8035 - val_sparse_categorical_accuracy: 0.0695
Epoch 2/6
469/469 [==============================] - 2s 4ms/step - loss: 1.3712 - sparse_categorical_accuracy: 0.0946 - val_loss: 1.1951 - val_sparse_categorical_accuracy: 0.0902
Epoch 3/6
469/469 [==============================] - 2s 4ms/step - loss: 0.9998 - sparse_categorical_accuracy: 0.0931 - val_loss: 0.9470 - val_sparse_categorical_accuracy: 0.0994
Epoch 4/6
469/469 [==============================] - 2s 4ms/step - loss: 0.8320 - sparse_categorical_accuracy: 0.0950 - val_loss: 0.8377 - val_sparse_categorical_accuracy: 0.0753
Epoch 5/6
469/469 [==============================] - 2s 4ms/step - loss: 0.7298 - sparse_categorical_accuracy: 0.0875 - val_loss: 0.7917 - val_sparse_categorical_accuracy: 0.0813
Epoch 6/6
469/469 [==============================] - 2s 4ms/step - loss: 0.6454 - sparse_categorical_accu

## Results

In [ ]:
result = ga.best_individual()
print(result)

neurons_in_layers = [convert_list_to_int(result[1][0]), convert_list_to_int(result[1][1])]

print(neurons_in_layers)  # Last run gave 245 and 128 neurons on first and second layer.

(25.57030714220471, [[1, 1, 1, 1, 0, 1, 0, 1], [1, 0, 0, 0, 0, 0, 0, 0]])
[245, 128]


# Debug

## Notes
Some data appear to have structure like `[[...], 0]` (`list[list, int]`) and not `[[...], [...]]` (`list[list, list]`). 

Individuals are generated correctly but thic corrupted data are passed to fitness function, so error is somwhere between.

For noe I do not know what causes this error.